# prep_artifacts — notebook-driven data preparation for XTrendLL

Run this on your **local machine** once; upload the resulting folder to Google Drive; then the two training notebooks (`xtrendll_a1_no_bennett.ipynb` and `xtrendll_a2_with_bennett.ipynb`) can skip CPD + feature engineering entirely.

`xtrendll` is self-contained: every helper this notebook uses (`build_panel`, `time_split`, `segment_panel_cached`, `build_regime_cache_cached`, `build_lead_lag_matrix_cached`, …) lives inside the package. No sibling repo needed.

**Stages** (each in its own cell, with `tqdm` where possible):
1. Imports + environment info
2. Run config (universe, date range, tag, CPD / Bennett options)
3. Download prices and build the feature panel
4. CPD regimes on the **train** slice (used as the training context pool)
5. Causal CPD regime cache for **val** dates
6. Causal CPD regime cache for **test** dates
7. Bennett lead-lag matrix on the train panel (**only for A2**; skip this cell for A1)
8. Save tagged pickles + MANIFEST.json into `./artifacts/`

In [10]:
# ── 1. Imports + environment info ───────────────────────────────────────
import os, sys, time, json, pickle, hashlib, platform
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Add the parent of this notebook to sys.path so `import xtrendll` works.
# When you run this from inside a clone of the xtrendll repo, the parent is
# the repo root.  Adjust if you keep notebooks elsewhere.
HERE = Path('.').resolve()
for candidate in (HERE.parent, HERE.parent.parent, HERE):
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from xtrendll.config import DATA, CPD, DEFAULT_TICKERS
from xtrendll.cpd import segment_panel_cached, build_regime_cache_cached
from xtrendll.data import build_panel, time_split
from xtrendll.lead_lag import build_lead_lag_matrix_cached, build_lag_ranking_cached
from xtrendll.prep_artifacts import UNIVERSES, TUNING_21, load_artifacts

print(f'python {sys.version.split()[0]}  |  numpy {np.__version__}  |  pandas {pd.__version__}')
print(f'cpu count : {os.cpu_count()}')
print(f'universes : {list(UNIVERSES.keys())}')

python 3.14.3  |  numpy 2.4.4  |  pandas 2.3.3
cpu count : 8
universes : ['default_50', 'tuning_21']


In [11]:
# ── 2. Run config ───────────────────────────────────────────────────────
# Edit these parameters and re-run the remaining cells.

UNIVERSE       = 'default_50'                # 'default_50' or 'tuning_21'
START          = '2008-01-01'
END            = pd.Timestamp.today().strftime('%Y-%m-%d')
TAG            = f'v1_etf{len(UNIVERSES[UNIVERSE])}'   # output file suffix

CPD_JOBS        = max(os.cpu_count() or 1, 1)
RECOMPUTE_EVERY = CPD['recompute_every']    # days between causal CPD snapshots

# A1  = False, False
# A2  = True,  False         (2-D Bennett only)
# A5  = True,  True          (A5 notebook needs the 3-D per-lag artefact)
BUILD_BENNETT       = True
BENNETT_MAX_LAG     = 5

BUILD_LAG_RANKINGS  = True                    # produces lag_rankings__<tag>.pkl (A2.5 / A3 / A5)
LAG_RANKING_LAGS    = (1, 5, 10, 21, 30)      # must match LL_RUN["lag_set"] in the A5 notebook
LAG_RANKING_TOP_K   = 5                       # peers kept per (target, lag) in the hard mask
LAG_RANKING_MIN_OBS = 252                     # minimum overlapping obs required for a pair

OUT_DIR      = Path('./artifacts_50').resolve()
HIDDEN_CACHE = OUT_DIR / '_cpd_cache'       # internal cpd.py cache — reuse across runs

OUT_DIR.mkdir(parents=True, exist_ok=True)
HIDDEN_CACHE.mkdir(exist_ok=True)

DATA_RUN = deepcopy(DATA)
DATA_RUN['tickers'] = list(UNIVERSES[UNIVERSE])
DATA_RUN['start']   = START
DATA_RUN['end']     = END

print(f'universe            : {UNIVERSE}  ({len(DATA_RUN["tickers"])} tickers)')
print(f'window              : {START}  →  {END}')
print(f'tag                 : {TAG}')
print(f'out_dir             : {OUT_DIR}')
print(f'CPD jobs            : {CPD_JOBS}')
print(f'recompute_every     : {RECOMPUTE_EVERY}')
print(f'build_bennett       : {BUILD_BENNETT}  (max_lag={BENNETT_MAX_LAG})')
print(f'build_lag_rankings  : {BUILD_LAG_RANKINGS}  (lags={LAG_RANKING_LAGS}, top_k={LAG_RANKING_TOP_K})')

universe        : default_50  (50 tickers)
window          : 2008-01-01  →  2026-04-22
tag             : v1_etf50
out_dir         : /Users/zhangziyao/Desktop/MSCF_course/intro_deep_learning/S11685_Final_Project/xtrendll/artifacts_50
CPD jobs        : 8
recompute_every : 21
build_bennett   : True  (max_lag=5)


In [12]:
# ── 3. Download + feature engineering: build_panel ──────────────────────
stage_bar = tqdm(total=6, desc='prep stages', position=0)
stage_bar.set_postfix_str('build_panel')

t0 = time.perf_counter()
panel, fcols, tk2id = build_panel(DATA_RUN)
panel['date'] = pd.to_datetime(panel['date'])
dt = time.perf_counter() - t0

print(f'  panel shape       = {panel.shape}')
print(f'  assets kept       = {len(tk2id)}  / {len(DATA_RUN["tickers"])} requested')
print(f'  feature cols      = {fcols}')
print(f'  date range        = {panel["date"].min().date()}  →  {panel["date"].max().date()}')
print(f'  elapsed           = {dt:.1f}s')

train_d, val_d, test_d = time_split(panel, DATA_RUN['train_frac'], DATA_RUN['val_frac'])
train_end = pd.Timestamp(train_d.max())
val_end   = pd.Timestamp(val_d.max())
test_end  = pd.Timestamp(test_d.max())
train_panel     = panel[panel['date'] <= train_end].copy()
val_panel_hist  = panel[panel['date'] <= val_end].copy()
test_panel_hist = panel[panel['date'] <= test_end].copy()
print(f'  train days = {len(train_d)}   val days = {len(val_d)}   test days = {len(test_d)}')

stage_bar.update(1)

prep stages:   0%|          | 0/6 [00:00<?, ?it/s]

  panel shape       = (214500, 14)
  assets kept       = 50  / 50 requested
  feature cols      = ['norm_ret_1', 'norm_ret_21', 'norm_ret_63', 'norm_ret_126', 'norm_ret_252', 'macd_8_24', 'macd_16_28', 'macd_32_96']
  date range        = 2009-03-31  →  2026-04-20
  elapsed           = 2.0s
  train days = 3003   val days = 643   test days = 644


True

In [13]:
# ── 4. CPD regimes on the train panel ───────────────────────────────────
# segment_panel_cached runs GP-based change-point detection per ticker,
# in parallel via ProcessPoolExecutor.  Per-ticker progress is printed by
# cpd.py when verbose=1.

stage_bar.set_postfix_str('train CPD')
t0 = time.perf_counter()
train_regimes = segment_panel_cached(
    train_panel, cache_dir=str(HIDDEN_CACHE),
    n_jobs=CPD_JOBS, verbose=1,
)
dt = time.perf_counter() - t0

reg_lengths = [re - rs + 1 for rr in train_regimes.values() for rs, re in rr]
print(f'  regimes discovered = {len(reg_lengths):,}  across {len(train_regimes)} tickers')
if reg_lengths:
    print(f'  regime length    min={min(reg_lengths)}  median={int(np.median(reg_lengths))}  max={max(reg_lengths)}')
print(f'  elapsed          = {dt:.1f}s')

stage_bar.update(1)

Running CPD across 50 tickers with 8 worker(s) ...
Cached train regimes to /Users/zhangziyao/Desktop/MSCF_course/intro_deep_learning/S11685_Final_Project/xtrendll/artifacts_50/_cpd_cache/regimes_1ef471700f150eae.pkl
  regimes discovered = 7,066  across 50 tickers
  regime length    min=6  median=22  max=38
  elapsed          = 72.8s


True

In [14]:
# ── 5. Causal CPD regime cache for val dates ────────────────────────────
stage_bar.set_postfix_str('val regime cache')
t0 = time.perf_counter()
val_regime_cache = build_regime_cache_cached(
    val_panel_hist, val_d,
    recompute_every=RECOMPUTE_EVERY,
    cache_dir=str(HIDDEN_CACHE),
    n_jobs=CPD_JOBS, verbose=1,
)
dt = time.perf_counter() - t0
print(f'  snapshots = {len(val_regime_cache)}')
print(f'  elapsed   = {dt:.1f}s')
stage_bar.update(1)

CPD snapshot 1/32: 2021-03-05
CPD snapshot 2/32: 2021-04-06
CPD snapshot 3/32: 2021-05-05
CPD snapshot 4/32: 2021-06-04
CPD snapshot 5/32: 2021-07-06
CPD snapshot 6/32: 2021-08-04
CPD snapshot 7/32: 2021-09-02
CPD snapshot 8/32: 2021-10-04
CPD snapshot 9/32: 2021-11-02
CPD snapshot 10/32: 2021-12-02
CPD snapshot 11/32: 2022-01-03
CPD snapshot 12/32: 2022-02-02
CPD snapshot 13/32: 2022-03-04
CPD snapshot 14/32: 2022-04-04
CPD snapshot 15/32: 2022-05-04
CPD snapshot 16/32: 2022-06-03
CPD snapshot 17/32: 2022-07-06
CPD snapshot 18/32: 2022-08-04
CPD snapshot 19/32: 2022-09-02
CPD snapshot 20/32: 2022-10-04
CPD snapshot 21/32: 2022-11-02
CPD snapshot 22/32: 2022-12-02
CPD snapshot 23/32: 2023-01-04
CPD snapshot 24/32: 2023-02-03
CPD snapshot 25/32: 2023-03-07
CPD snapshot 26/32: 2023-04-05
CPD snapshot 27/32: 2023-05-05
CPD snapshot 28/32: 2023-06-06
CPD snapshot 29/32: 2023-07-07
CPD snapshot 30/32: 2023-08-07
CPD snapshot 31/32: 2023-09-06
CPD snapshot 32/32: 2023-09-22
Cached regime sna

True

In [15]:
# ── 6. Causal CPD regime cache for test dates ───────────────────────────
stage_bar.set_postfix_str('test regime cache')
t0 = time.perf_counter()
test_regime_cache = build_regime_cache_cached(
    test_panel_hist, test_d,
    recompute_every=RECOMPUTE_EVERY,
    cache_dir=str(HIDDEN_CACHE),
    n_jobs=CPD_JOBS, verbose=1,
)
dt = time.perf_counter() - t0
print(f'  snapshots = {len(test_regime_cache)}')
print(f'  elapsed   = {dt:.1f}s')
stage_bar.update(1)

CPD snapshot 1/32: 2023-09-25
CPD snapshot 2/32: 2023-10-24
CPD snapshot 3/32: 2023-11-22
CPD snapshot 4/32: 2023-12-22
CPD snapshot 5/32: 2024-01-25
CPD snapshot 6/32: 2024-02-26
CPD snapshot 7/32: 2024-03-26
CPD snapshot 8/32: 2024-04-25
CPD snapshot 9/32: 2024-05-24
CPD snapshot 10/32: 2024-06-26
CPD snapshot 11/32: 2024-07-26
CPD snapshot 12/32: 2024-08-26
CPD snapshot 13/32: 2024-09-25
CPD snapshot 14/32: 2024-10-24
CPD snapshot 15/32: 2024-11-22
CPD snapshot 16/32: 2024-12-24
CPD snapshot 17/32: 2025-01-28
CPD snapshot 18/32: 2025-02-27
CPD snapshot 19/32: 2025-03-28
CPD snapshot 20/32: 2025-04-29
CPD snapshot 21/32: 2025-05-29
CPD snapshot 22/32: 2025-06-30
CPD snapshot 23/32: 2025-07-30
CPD snapshot 24/32: 2025-08-28
CPD snapshot 25/32: 2025-09-29
CPD snapshot 26/32: 2025-10-28
CPD snapshot 27/32: 2025-11-26
CPD snapshot 28/32: 2025-12-29
CPD snapshot 29/32: 2026-01-29
CPD snapshot 30/32: 2026-03-02
CPD snapshot 31/32: 2026-03-31
CPD snapshot 32/32: 2026-04-20
Cached regime sna

True

In [16]:
# ── 7. Bennett lead-lag matrix + per-lag rankings ──────────────────────
# Both are fit on the train panel only (causal), matching train_regimes.

stage_bar.set_postfix_str('bennett')
lead_lag_payload = None
if BUILD_BENNETT:
    t0 = time.perf_counter()
    lead_lag_payload = build_lead_lag_matrix_cached(
        train_panel, cache_dir=str(HIDDEN_CACHE),
        max_lag=BENNETT_MAX_LAG, verbose=1,
    )
    dt = time.perf_counter() - t0
    S = lead_lag_payload['S']
    print(f'  2-D Bennett S shape  = {S.shape}')
    print(f'  edge density         = {(S > 0).mean():.2%}')
    print(f'  max edge weight      = {S.max():.3f}')
    print(f'  elapsed              = {dt:.1f}s')
else:
    print('  Bennett build SKIPPED (BUILD_BENNETT=False).  A2 / A5 notebooks will refuse to run.')

lag_ranking_payload = None
if BUILD_LAG_RANKINGS:
    t0 = time.perf_counter()
    lag_ranking_payload = build_lag_ranking_cached(
        train_panel, train_d, tk2id,
        cache_dir=str(HIDDEN_CACHE),
        lags=LAG_RANKING_LAGS, top_k=LAG_RANKING_TOP_K,
        min_obs=LAG_RANKING_MIN_OBS, verbose=1,
    )
    dt = time.perf_counter() - t0
    n_tk = len(lag_ranking_payload['tickers'])
    print(f'  3-D per-lag stack    = {len(lag_ranking_payload["lags"])} lags × {n_tk}×{n_tk}')
    print(f'  top_k per target/lag = {lag_ranking_payload["top_k"]}')
    print(f'  elapsed              = {dt:.1f}s')
else:
    print('  Lag-ranking build SKIPPED (BUILD_LAG_RANKINGS=False).  A5 notebook will refuse to run.')

stage_bar.update(1)

Building lead-lag matrix (max_lag=5) ...


lead-lag CCF-AUC:   0%|          | 0/5 [00:00<?, ?it/s]

Cached lead-lag matrix to /Users/zhangziyao/Desktop/MSCF_course/intro_deep_learning/S11685_Final_Project/xtrendll/artifacts_50/_cpd_cache/lead_lag_5bb9f7780a6b85ce.pkl
  S shape         = (50, 50)
  edge density    = 49.00%
  max edge weight = 0.859
  elapsed         = 0.1s


True

In [17]:
# ── 8. Save tagged pickles + MANIFEST.json ──────────────────────────────
stage_bar.set_postfix_str('write artefacts')

def _dump(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

def _sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

panel_bundle = {
    'panel': panel, 'fcols': list(fcols),
    'tk2id': dict(tk2id), 'data_run': DATA_RUN,
}

n_writes = 4 + int(BUILD_BENNETT and lead_lag_payload is not None) + int(BUILD_LAG_RANKINGS and lag_ranking_payload is not None)

paths = {}
with tqdm(total=n_writes, desc='writing pickles', leave=False) as wbar:
    paths['panel_bundle']      = OUT_DIR / f'panel_bundle__{TAG}.pkl';       _dump(panel_bundle,       paths['panel_bundle']);     wbar.update(1)
    paths['train_regimes']     = OUT_DIR / f'train_regimes__{TAG}.pkl';      _dump(train_regimes,      paths['train_regimes']);    wbar.update(1)
    paths['val_regime_cache']  = OUT_DIR / f'val_regime_cache__{TAG}.pkl';   _dump(val_regime_cache,   paths['val_regime_cache']); wbar.update(1)
    paths['test_regime_cache'] = OUT_DIR / f'test_regime_cache__{TAG}.pkl';  _dump(test_regime_cache,  paths['test_regime_cache']);wbar.update(1)
    if BUILD_BENNETT and lead_lag_payload is not None:
        paths['lead_lag_matrix'] = OUT_DIR / f'lead_lag_matrix__{TAG}.pkl'
        _dump(lead_lag_payload, paths['lead_lag_matrix']);                                                                          wbar.update(1)
    if BUILD_LAG_RANKINGS and lag_ranking_payload is not None:
        paths['lag_rankings'] = OUT_DIR / f'lag_rankings__{TAG}.pkl'
        _dump(lag_ranking_payload, paths['lag_rankings']);                                                                          wbar.update(1)

manifest = {
    'tag': TAG, 'universe': UNIVERSE, 'tickers': DATA_RUN['tickers'],
    'start': START, 'end': END,
    'cpd_config': dict(CPD), 'recompute_every': RECOMPUTE_EVERY,
    'bennett': bool(BUILD_BENNETT),
    'bennett_max_lag': int(BENNETT_MAX_LAG) if BUILD_BENNETT else None,
    'lag_rankings': bool(BUILD_LAG_RANKINGS),
    'lag_ranking_lags': list(LAG_RANKING_LAGS) if BUILD_LAG_RANKINGS else None,
    'lag_ranking_top_k': int(LAG_RANKING_TOP_K) if BUILD_LAG_RANKINGS else None,
    'lag_ranking_min_obs': int(LAG_RANKING_MIN_OBS) if BUILD_LAG_RANKINGS else None,
    'n_assets_kept': len(tk2id), 'feature_cols': list(fcols),
    'train_days': int(len(train_d)), 'val_days': int(len(val_d)), 'test_days': int(len(test_d)),
    'hostname': platform.node(),
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'files': {
        key: {'name': p.name, 'sha256': _sha256_file(p), 'bytes': p.stat().st_size}
        for key, p in paths.items()
    },
}
with open(OUT_DIR / 'MANIFEST.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)

stage_bar.update(1)
stage_bar.close()

print(f'\nArtefacts written to {OUT_DIR}:')
for name in sorted(OUT_DIR.iterdir()):
    if name.is_file():
        mb = name.stat().st_size / (1 << 20)
        print(f'  {name.name:<40s}  {mb:7.2f} MB')
print('\nNext step: upload the whole folder (MANIFEST.json included; `_cpd_cache/` optional) to Google Drive,')
print(f'e.g. MyDrive/xtrendll_artifacts_{TAG}/, then open xtrendll_a1_no_bennett.ipynb / a2 / a5 on Colab.')

writing pickles:   0%|          | 0/5 [00:00<?, ?it/s]


Artefacts written to /Users/zhangziyao/Desktop/MSCF_course/intro_deep_learning/S11685_Final_Project/xtrendll/artifacts_50:
  MANIFEST.json                                0.00 MB
  lead_lag_matrix__v1_etf50.pkl                0.01 MB
  panel_bundle__v1_etf50.pkl                  21.69 MB
  test_regime_cache__v1_etf50.pkl              2.26 MB
  train_regimes__v1_etf50.pkl                  0.05 MB
  val_regime_cache__v1_etf50.pkl               1.89 MB

Next step: upload the whole folder (MANIFEST.json included; `_cpd_cache/` optional) to Google Drive,
e.g. MyDrive/xtrendll_artifacts_v1_etf50/, then open xtrendll_a1_no_bennett.ipynb (or a2) on Colab.


In [18]:
# ── 9. Optional sanity reload — confirms load_artifacts works ──────────
art = load_artifacts(str(OUT_DIR))
print(f'load_artifacts OK — tag={art["tag"]}  assets={len(art["tk2id"])}  '
      f'features={len(art["fcols"])}  val_snaps={len(art["val_regime_cache"])}  '
      f'test_snaps={len(art["test_regime_cache"])}  bennett={art["lead_lag_matrix"] is not None}')

[load_artifacts] tag=v1_etf50  path=/Users/zhangziyao/Desktop/MSCF_course/intro_deep_learning/S11685_Final_Project/xtrendll/artifacts_50
  panel rows=214,500  assets=50  features=8
  train_regimes tickers=50  val snapshots=32  test snapshots=32
  lead_lag_matrix shape=(50, 50)
load_artifacts OK — tag=v1_etf50  assets=50  features=8  val_snaps=32  test_snaps=32  bennett=True
